# Training a model in PyTorch, line by line

A recap of the Week 9 PyTorch hours: we train the character MLP language model on 240 names.
Run the cells in order with **Shift+Enter**. Each cell is a few lines; read them, then look at what they print.
Change a number and run the cell again to see what changes.

## 0. The data and the model (from Week 9)

Run these two cells first. They make the training examples (three characters → the next character) and the model.

In [ ]:
import math, os, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

if not os.path.exists("names.txt"):
    urllib.request.urlretrieve("https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/names.txt", "names.txt")
names = sorted(set(open("names.txt").read().split()))
perm = torch.randperm(len(names), generator=torch.Generator().manual_seed(0)).tolist()
n_tr = int(0.85 * len(names))
train_words = [names[i] for i in perm[:n_tr]]
val_words = [names[i] for i in perm[n_tr:]]

chars = ["."] + sorted(set("".join(names)))
stoi = {c: i for i, c in enumerate(chars)}

def make_examples(words, block=3):
    xs, ys = [], []
    for w in words:
        ctx = [0] * block
        for ch in w + ".":
            xs.append(ctx); ys.append(stoi[ch])
            ctx = ctx[1:] + [stoi[ch]]
    return torch.tensor(xs), torch.tensor(ys)

Xtr, Ytr = make_examples(train_words)
Xva, Yva = make_examples(val_words)
print("training examples:", Xtr.shape, " validation examples:", Xva.shape)
print("first example:", Xtr[0].tolist(), "->", Ytr[0].item())

In [ ]:
class MLPLM(nn.Module):
    def __init__(self, V=27, d=2, block=3, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.hidden = nn.Linear(block * d, hidden)
        self.out = nn.Linear(hidden, V)
    def forward(self, ctx):
        h = torch.tanh(self.hidden(self.emb(ctx).flatten(1)))
        return self.out(h)

torch.manual_seed(0)
lm = MLPLM()
print("parameters:", sum(p.numel() for p in lm.parameters()))

## 1. A minibatch

64 random examples out of the training set.

In [ ]:
torch.manual_seed(1)
idx = torch.randint(0, len(Xtr), (64,))
xb, yb = Xtr[idx], Ytr[idx]
print(xb.shape, yb.shape)

## 2. The forward pass and the loss

In [ ]:
logits = lm(xb)
print(logits.shape)

In [ ]:
loss = F.cross_entropy(logits, yb)
print(loss.item(), " ln 27 =", math.log(27))

## 3. The gradients

`backward()` fills the `.grad` field of every parameter.

In [ ]:
print(lm.out.bias.grad)

In [ ]:
loss.backward()
print(lm.out.bias.grad[:5])

## 4. One step of SGD

The optimizer holds the parameters and the learning rate. `step()` does $w \leftarrow w - \eta\,\nabla L$.

In [ ]:
opt = torch.optim.SGD(lm.parameters(), lr=0.3)
before = lm.out.bias[:3].detach().clone()
opt.step()
print(before)
print(before - 0.3 * lm.out.bias.grad[:3])
print(lm.out.bias[:3].detach())

In [ ]:
print(F.cross_entropy(lm(xb), yb).item())

## 5. Clearing the gradients

A second `backward()` adds to `.grad`. `zero_grad()` clears it.

In [ ]:
g = lm.out.bias.grad[0].item()
loss = F.cross_entropy(lm(xb), yb)
loss.backward()
print(g, lm.out.bias.grad[0].item())
opt.zero_grad()
print(lm.out.bias.grad)

## 6. The training loop

The same steps, repeated: a minibatch, the loss, `zero_grad`, `backward`, `step`.

In [ ]:
torch.manual_seed(0)
lm = MLPLM()
opt = torch.optim.SGD(lm.parameters(), lr=0.3)
for step in range(1, 1001):
    idx = torch.randint(0, len(Xtr), (64,))
    loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 200 == 0:
        print(step, round(loss.item(), 3))

## 7. Epochs

One epoch is one pass over the training examples.

In [ ]:
print(len(Xtr), "examples /", 64, "per minibatch =", len(Xtr) / 64, "steps per epoch")
print("1,000 steps =", round(1000 * 64 / len(Xtr), 1), "epochs")

## 8. The training and validation loss

`torch.no_grad()` computes the loss without recording a graph.

In [ ]:
with torch.no_grad():
    print("training:  ", round(F.cross_entropy(lm(Xtr), Ytr).item(), 3))
    print("validation:", round(F.cross_entropy(lm(Xva), Yva).item(), 3))

In [ ]:
for step in range(1001, 4001):
    idx = torch.randint(0, len(Xtr), (64,))
    loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 1000 == 0:
        with torch.no_grad():
            tr = F.cross_entropy(lm(Xtr), Ytr).item()
            va = F.cross_entropy(lm(Xva), Yva).item()
        print(step, " training", round(tr, 3), " validation", round(va, 3))

## 9. The learning rate

The same 1,000 steps from the same start, with three learning rates.

In [ ]:
for lr in [0.03, 0.3, 3.0]:
    torch.manual_seed(0)
    m = MLPLM()
    o = torch.optim.SGD(m.parameters(), lr=lr)
    for step in range(1000):
        idx = torch.randint(0, len(Xtr), (64,))
        loss = F.cross_entropy(m(Xtr[idx]), Ytr[idx])
        o.zero_grad(); loss.backward(); o.step()
    with torch.no_grad():
        print("lr", lr, " training loss", round(F.cross_entropy(m(Xtr), Ytr).item(), 3))

## 10. Sampling names

Start from "...", draw the next character from the model's probabilities, and stop at ".".

In [ ]:
g = torch.Generator().manual_seed(1)
for _ in range(8):
    ctx, word = [0, 0, 0], ""
    while True:
        with torch.no_grad():
            p = F.softmax(lm(torch.tensor([ctx])), dim=1)
        i = torch.multinomial(p[0], 1, generator=g).item()
        if i == 0 or len(word) > 15:
            break
        word += chars[i]
        ctx = ctx[1:] + [i]
    print(word)